In [44]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

padding=1 + kernel=3 + stride=1 조합은 spatial size 를 그대로 유지 하는 "same padding" 효과이다. 그래서 conv 에서는 크기 안 줄고, pool 두 번 거치면서만 28 → 14 → 7 로 줄어든다. 그래서 최종 feature map 이 16 채널 × 7×7 이고, flatten 하면 16*7*7 = 784.

In [45]:
class CNN(nn.Module):
    def __init__(self, in_channels = 1, num_classes = 10):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=(3, 3), stride=(1,1), padding=(1,1))
        self.pool = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=(3,3), stride=(1,1), padding=(1,1))
        self.fc1 = nn.Linear(16*7*7, num_classes)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x) 
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        return x
        

In [46]:
model = CNN()
x = torch.randn(64, 1, 28, 28)
model(x).shape

torch.Size([64, 10])

In [47]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device 

device(type='cpu')

In [48]:
in_channel = 1
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 1

In [49]:
train_dataset = datasets.MNIST(root='dataset/', train=True, transform = transforms.ToTensor(), download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = datasets.MNIST(root='dataset/', train=False, transform = transforms.ToTensor(), download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [50]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [51]:
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device) 
        targets = targets.to(device)
        
        # get to corret image
        # data = data.reshape(data.shape[0], -1)
        
        # forward
        scores = model(data)
        loss = criterion(scores, targets)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [52]:
def check_accuracy(loader, model):
    if loader.dataset.train:
        print('checking accuracy on training data')
    else:
        print('checking accuracy on test data')
    
    num_correct = 0
    num_samples = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            # x = x.reshape(x.shape[0], -1)
            
            scores = model(x)
            _, predictions = scores.max(dim=1)     # find max value in class dimension, result shape is (64,)
            num_correct += (predictions == y).sum().item()    # 0 dim Tensor to int
            num_samples += predictions.size(0) 
        
        print(f'Got (num_correct) / (num_samples) with accuracy {float(num_correct)/float(num_samples)*100:.2f}')    
    
    model.train()    


In [53]:
check_accuracy(test_loader, model)
check_accuracy(train_loader, model)

checking accuracy on test data
Got (num_correct) / (num_samples) with accuracy 96.66
checking accuracy on training data
Got (num_correct) / (num_samples) with accuracy 96.33
